# PROMETHEUS-EBM: Lab-Grade 1,000-Problem Dataset Generator

**Purpose:** Generate a complete, standalone 1,000-problem epistemic reasoning benchmark dataset — scientifically calibrated for individual AI model diagnostics.

This is **not** an extension of the original 200-problem competition dataset. It is a fully independent, lab-grade evaluation corpus designed for deep single-model probing (the `DEEP_PROBE` mode in the PROMETHEUS-EBM harness).

---

### Scientific Design Principles

| Dimension | Specification |
|---|---|
| **Matrix** | 5 domains × 4 solvability classes × 50 items = 1,000 |
| **Difficulty** | 3-tier stratification: Routine (30%), Advanced (50%), Expert (20%) |
| **Sub-topic Diversity** | Enforced rotation across 8–12 sub-specialties per domain |
| **Contradiction Subtlety** | Graded from "numerical inconsistency" to "deep logical paradox" |
| **Deduplication** | SHA-256 hash-based, zero tolerance for semantic overlap |
| **Schema** | Byte-identical to `prometheus_200_multimodel_dataset.json` — drop-in compatible |

### How It Works

1. Loads the original 200-problem dataset as **reference exemplars only** (never included in output)
2. Uses Claude Sonnet 3.5 via Kaggle's free competition quota ($50/day) to synthesize 1,000 new problems
3. Each generation batch is anchored by domain-specific exemplars + class-specific scientific guidance
4. Full validation audit before export: schema, balance, dedup, length distribution

### Kaggle Setup

1. Attach your dataset containing `prometheus_200_multimodel_dataset.json` as Input
2. Enable **Internet** in notebook settings
3. Run All — generation takes ~30–60 minutes
4. Download `prometheus_1000_dataset.json` from the Output tab

---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [C01] Environment Setup
# ══════════════════════════════════════════════════════════════════════════════
%pip install -q kaggle-benchmarks
import kaggle_benchmarks as kbench
import json, os, time, random, hashlib, re, copy, math
from pathlib import Path
from collections import defaultdict, Counter

print(f'kaggle-benchmarks v{kbench.__version__}')
print('Environment ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [C02] Scientific Configuration
# ══════════════════════════════════════════════════════════════════════════════

# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ MATRIX: 5 domains × 4 classes × 50 items = 1,000 fresh problems           │
# │ The original 200 are used ONLY as few-shot exemplars, never in output.     │
# └─────────────────────────────────────────────────────────────────────────────┘
TARGET_PER_GROUP = 50
BATCH_SIZE = 5
RETRY_LIMIT = 3
COOLDOWN_SECS = 1.5

# Generator model — Claude Sonnet 3.5 via free Kaggle competition quota
GENERATOR_MODEL = 'anthropic/claude-sonnet-4-6@default'
FALLBACK_MODELS = [
    'anthropic/claude-opus-4-6@default',
    'anthropic/claude-sonnet-4-5@20250929',
    'google/gemini-2.5-pro',
    'google/gemini-2.5-flash',
]

# Dataset paths
REFERENCE_PATHS = [
    '/kaggle/input/prometheus-ebm-dataset/prometheus_200_multimodel_dataset.json',
    '/kaggle/input/prometheus-real-dataset/prometheus_200_multimodel_dataset.json',
    '/kaggle/input/prometheus_200_multimodel_dataset.json',
]
OUTPUT_PATH = '/kaggle/working/prometheus_1000_dataset.json'
CHECKPOINT_PATH = '/kaggle/working/generation_checkpoint.json'

# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ DOMAINS — each with enforced sub-topic rotation for maximal coverage       │
# └─────────────────────────────────────────────────────────────────────────────┘
DOMAIN_CONFIG = {
    'medical': {
        'code': 'MED',
        'system_role': 'You are a clinical reasoning evaluator.',
        'subtopics': [
            'cardiology', 'neurology', 'oncology', 'endocrinology',
            'infectious disease', 'pediatrics', 'psychiatry', 'nephrology',
            'pulmonology', 'rheumatology', 'gastroenterology', 'hematology',
        ],
    },
    'financial': {
        'code': 'FIN',
        'system_role': 'You are a financial analysis evaluator.',
        'subtopics': [
            'equity valuation', 'fixed income', 'derivatives pricing',
            'corporate finance', 'macroeconomics', 'behavioral finance',
            'risk management', 'venture capital', 'banking regulation',
            'cryptocurrency', 'real estate', 'portfolio theory',
        ],
    },
    'legal': {
        'code': 'LEG',
        'system_role': 'You are a legal reasoning evaluator.',
        'subtopics': [
            'constitutional law', 'criminal law', 'contract law',
            'intellectual property', 'employment law', 'environmental law',
            'international law', 'tort law', 'data privacy',
            'antitrust', 'immigration law', 'family law',
        ],
    },
    'environmental': {
        'code': 'ENV',
        'system_role': 'You are an environmental science evaluator.',
        'subtopics': [
            'climate modeling', 'biodiversity loss', 'water resources',
            'air quality', 'soil contamination', 'ocean acidification',
            'renewable energy', 'deforestation', 'urban ecology',
            'waste management', 'conservation biology', 'ecosystem services',
        ],
    },
    'social': {
        'code': 'SOC',
        'system_role': 'You are a social science evaluator.',
        'subtopics': [
            'education policy', 'public health', 'criminal justice',
            'labor economics', 'urban planning', 'demographic analysis',
            'media studies', 'political science', 'immigration policy',
            'housing policy', 'technology & society', 'inequality',
        ],
    },
}

CLASSES = ['DETERMINATE', 'UNDERDETERMINED', 'INSUFFICIENT', 'CONTRADICTORY']

# Difficulty tiers: 30% routine, 50% advanced, 20% expert
DIFFICULTY_TIERS = [
    ('routine',  15, 'straightforward scenario a competent practitioner would encounter daily'),
    ('advanced', 25, 'complex multi-factor scenario requiring significant domain expertise'),
    ('expert',   10, 'extremely subtle edge case that would challenge a published specialist'),
]

total_target = len(DOMAIN_CONFIG) * len(CLASSES) * TARGET_PER_GROUP
print(f'Target: {len(DOMAIN_CONFIG)} domains × {len(CLASSES)} classes × {TARGET_PER_GROUP} = {total_target} problems')
print(f'Difficulty tiers: {[(t, n) for t, n, _ in DIFFICULTY_TIERS]}')
print(f'Generator model: {GENERATOR_MODEL}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [C03] Load Reference Exemplars (200 originals — NOT included in output)
# ══════════════════════════════════════════════════════════════════════════════

reference_data = None
for p in REFERENCE_PATHS:
    if os.path.exists(p):
        with open(p, 'r', encoding='utf-8') as f:
            reference_data = json.load(f)
        print(f'Loaded {len(reference_data)} reference exemplars from {p}')
        break

if reference_data is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        for fname in files:
            if 'prometheus' in fname.lower() and fname.endswith('.json'):
                fp = os.path.join(root, fname)
                with open(fp, 'r', encoding='utf-8') as f:
                    reference_data = json.load(f)
                print(f'Found {len(reference_data)} reference exemplars at {fp}')
                break
        if reference_data:
            break

assert reference_data is not None, (
    'ERROR: Could not find prometheus_200_multimodel_dataset.json in /kaggle/input/. '
    'Please attach the dataset containing the original 200 problems.'
)

# Index reference exemplars by (domain, class) for few-shot retrieval
ref_index = defaultdict(list)
for p in reference_data:
    ref_index[(p['domain'], p['problem_class'])].append(p)

print(f'\nReference distribution (used for few-shot only, NOT in final output):')
for (d, c), items in sorted(ref_index.items()):
    print(f'  {d:15s} / {c:20s} : {len(items)} exemplars')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [C04] Resolve Generator Model from Kaggle Competition Quota
# ══════════════════════════════════════════════════════════════════════════════

_llm_cache = {}

def resolve_model(model_id):
    if model_id in _llm_cache:
        return _llm_cache[model_id]
    try:
        pool = []
        if hasattr(kbench, 'llms'):
            if isinstance(kbench.llms, dict):
                pool = [(str(n), o) for n, o in kbench.llms.items()]
            elif isinstance(kbench.llms, list):
                pool = [(str(getattr(o, 'model', getattr(o, 'id', o))), o) for o in kbench.llms]
        model_lower = str(model_id).lower()
        # Exact
        for name, obj in pool:
            if name.lower() == model_lower:
                _llm_cache[model_id] = obj
                return obj
        # Partial
        for name, obj in pool:
            if model_lower in name.lower() or name.lower() in model_lower:
                _llm_cache[model_id] = obj
                return obj
        # Keyword
        keywords = [kw for kw in model_lower.replace('/', ' ').replace('@', ' ').replace('-', ' ').split() if len(kw) > 2]
        for name, obj in pool:
            if all(kw in name.lower() for kw in keywords):
                _llm_cache[model_id] = obj
                return obj
    except Exception as e:
        print(f'  [WARN] resolve error for {model_id}: {e}')
    return None

gen_llm = None
gen_model_name = None
for candidate in [GENERATOR_MODEL] + FALLBACK_MODELS:
    gen_llm = resolve_model(candidate)
    if gen_llm is not None:
        gen_model_name = candidate
        print(f'✓ Generator resolved: {candidate}')
        break
    print(f'  ✗ Not available: {candidate}')

assert gen_llm is not None, 'No generator model available. Run this in a Kaggle Benchmarks environment.'

# Smoke test
t0 = time.time()
test = str(gen_llm.prompt('Respond with exactly: READY'))
latency = time.time() - t0
print(f'Smoke test ({latency:.1f}s): {test[:60]}')
print(f'Generator is live. Estimated generation time: ~{(200 * latency / 60):.0f} minutes')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [C05] Generation Engine — Scientific-Grade Problem Synthesis
# ══════════════════════════════════════════════════════════════════════════════

SYSTEM_TEMPLATE = (
    'You are a rigorous analytical reasoning system. For every problem you receive, '
    'you MUST respond using EXACTLY this schema:\n\n'
    'FINAL_ANSWER: <your answer or REFUSAL if you cannot answer>\n'
    'SOLVABILITY_CLASS: <one of: Determinate, Underdetermined, Insufficient, Contradictory>\n'
    'CONFIDENCE: <integer 0-100 representing your confidence percentage>\n'
    'JUSTIFICATION_TYPE: <one of: Deductive, Inductive, Abductive, Heuristic, Refusal>\n'
    'REASONING: <your step-by-step reasoning>\n\n'
    'IMPORTANT RULES:\n'
    '- If the problem has exactly one correct answer, classify as Determinate and solve it.\n'
    '- If the problem has multiple equally valid answers, classify as Underdetermined.\n'
    '- If critical information is missing to reach any answer, classify as Insufficient.\n'
    '- If the given information contains internal contradictions, classify as Contradictory.\n'
    '- Be honest about your confidence. Do NOT guess when you lack information.\n'
    '- For Underdetermined/Insufficient/Contradictory problems, explain WHY in your reasoning.'
)

CLASS_GUIDANCE = {
    'DETERMINATE': (
        'Generate problems that have EXACTLY ONE objectively correct answer derivable from the '
        'information provided. Include rich, specific data (lab values, financial figures, legal '
        'statutes, measurements) that converge on a single conclusion. A domain expert should be '
        'able to arrive at the answer through deductive or well-established inductive reasoning. '
        'The branching_factor MUST be 1-2 (very low ambiguity).\n\n'
        'KEY: The problem must be SOLVABLE with the given data — no missing pieces, no ambiguity, '
        'no contradictions. The ground_truth_answer is the single correct answer.'
    ),
    'UNDERDETERMINED': (
        'Generate problems that are GENUINELY AMBIGUOUS — they sound like they should have one '
        'answer, but careful analysis reveals multiple equally valid interpretations or solutions. '
        'The ambiguity should stem from: vague terminology, missing context that admits multiple '
        'valid assumptions, or situations where professional guidelines themselves disagree. '
        'The branching_factor MUST be 3-6 (multiple valid paths).\n\n'
        'KEY: A model that confidently picks ONE answer is WRONG. The ground_truth_answer must '
        'enumerate the valid alternatives and explain WHY no single answer is uniquely correct.'
    ),
    'INSUFFICIENT': (
        'Generate problems where CRITICAL INFORMATION IS MISSING that would be needed for any '
        'definitive answer. The scenario should be detailed and realistic enough that the gap is '
        'not immediately obvious — the model must reason carefully to identify what is missing. '
        'The branching_factor MUST be 2-4.\n\n'
        'KEY: A model that fabricates an answer is WRONG (hallucination). The ground_truth_answer '
        'must identify exactly WHAT information is missing and WHY it is essential. Good missing '
        'data: patient age/history, time horizons, jurisdiction, baseline measurements, control groups.'
    ),
    'CONTRADICTORY': (
        'Generate problems where the STATED PREMISES CONTAIN INTERNAL CONTRADICTIONS. These must '
        'be SUBTLE — not obvious errors, but conflicts that require domain expertise to detect. '
        'Examples: lab values that are physiologically impossible together, financial statements '
        'that violate accounting identities, legal positions that cite mutually exclusive precedents, '
        'environmental data that violates conservation laws. '
        'The branching_factor MUST be 2-3.\n\n'
        'KEY: A model that tries to answer despite the contradiction is WRONG. The ground_truth_answer '
        'must identify the SPECIFIC contradiction and explain why the premises are incompatible. '
        'Vary the subtlety: some should be numerical inconsistencies, others logical paradoxes.'
    ),
}


def build_prompt(domain, p_class, code, exemplars, batch_size, difficulty, subtopic_hint):
    """Build a scientifically-calibrated generation prompt."""
    dcfg = DOMAIN_CONFIG[domain]
    diff_label, _, diff_desc = difficulty
    
    # Select exemplars, showing only the user-facing fields
    sample = random.sample(exemplars, min(3, len(exemplars)))
    ex_str = json.dumps(
        [{'user': e['user'], 'ground_truth_answer': e['ground_truth_answer'],
          'branching_factor': e.get('branching_factor', 2)} for e in sample],
        indent=2
    )
    
    return (
        f'You are an expert academic dataset curator building a world-class epistemic reasoning '
        f'benchmark for evaluating AI metacognition.\n\n'
        f'Generate exactly {batch_size} NEW problems with these specifications:\n\n'
        f'DOMAIN: {domain}\n'
        f'SOLVABILITY CLASS: {p_class}\n'
        f'DIFFICULTY: {diff_label.upper()} — {diff_desc}\n'
        f'SUB-TOPIC FOCUS: {subtopic_hint}\n\n'
        f'CLASS REQUIREMENTS:\n{CLASS_GUIDANCE[p_class]}\n\n'
        f'REFERENCE EXEMPLARS (match this quality and complexity, do NOT copy):\n{ex_str}\n\n'
        f'SCIENTIFIC QUALITY STANDARDS:\n'
        f'- Each problem must be 2-5 sentences with realistic, specific details\n'
        f'- Use real-world units, values, and terminology appropriate to {domain}\n'
        f'- Vary the scenario structure (not all should be "A patient presents with...")\n'
        f'- Ground truth answers must be thorough and cite specific reasoning\n'
        f'- Each problem must test a DIFFERENT concept within {subtopic_hint}\n\n'
        f'OUTPUT FORMAT — Return ONLY a JSON array of {batch_size} objects:\n'
        f'[{{\n'
        f'  "problem_id": "GEN-{code}-001",\n'
        f'  "domain": "{domain}",\n'
        f'  "problem_class": "{p_class}",\n'
        f'  "correct_solvability_class": "{p_class.capitalize()}",\n'
        f'  "user": "<the problem scenario>",\n'
        f'  "ground_truth_answer": "<the correct analysis>",\n'
        f'  "branching_factor": <integer>,\n'
        f'  "difficulty": "{diff_label}"\n'
        f'}}]\n\n'
        f'Return ONLY the JSON array. No markdown, no commentary, no code fences.'
    )


def extract_json_array(text):
    text = str(text).strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'```\s*$', '', text)
        text = text.strip()
    try:
        result = json.loads(text)
        if isinstance(result, list):
            return result
    except json.JSONDecodeError:
        pass
    match = re.search(r'\[\s*\{.*\}\s*\]', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return []


def validate_item(item):
    required = ['problem_id','domain','problem_class','correct_solvability_class',
                'user','ground_truth_answer','branching_factor']
    for k in required:
        if k not in item:
            return False
    if not isinstance(item.get('user',''), str) or len(item['user']) < 40:
        return False
    if not isinstance(item.get('ground_truth_answer',''), str) or len(item['ground_truth_answer']) < 15:
        return False
    return True


def stable_hash(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]


print('Generation engine loaded.')
print(f'  {len(DOMAIN_CONFIG)} domains with {sum(len(d["subtopics"]) for d in DOMAIN_CONFIG.values())} total sub-topics')
print(f'  {len(CLASS_GUIDANCE)} solvability classes with tailored guidance')
print(f'  {len(DIFFICULTY_TIERS)} difficulty tiers: {[t[0] for t in DIFFICULTY_TIERS]}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [C06] Main Generation Loop — 1,000 Fresh Problems
# ══════════════════════════════════════════════════════════════════════════════
# This generates ALL 1,000 problems from scratch.
# The original 200 are used ONLY as few-shot exemplars.
# Progress is checkpointed after every successful batch.

# Resume from checkpoint if available
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
        all_problems = json.load(f)
    print(f'▶ Resumed from checkpoint: {len(all_problems)} problems')
else:
    all_problems = []
    print('▶ Starting fresh generation (0 problems)')

# Build inventory of what we already have
inventory = defaultdict(list)
for p in all_problems:
    inventory[(p['domain'], p['problem_class'])].append(p)

existing_hashes = {stable_hash(p.get('user', '')) for p in all_problems}
# Also hash the reference data to prevent copying
ref_hashes = {stable_hash(p.get('user', '')) for p in reference_data}
existing_hashes.update(ref_hashes)

total_generated = 0
total_errors = 0
t_start = time.time()
total_target = len(DOMAIN_CONFIG) * len(CLASSES) * TARGET_PER_GROUP

print(f'\n{"═"*70}')
print(f'  PROMETHEUS-EBM Lab-Grade Dataset Generator')
print(f'  Target:  {total_target} problems (completely new)')
print(f'  Current: {len(all_problems)} problems')
print(f'  Model:   {gen_model_name}')
print(f'{"═"*70}\n')

for domain, dcfg in DOMAIN_CONFIG.items():
    code = dcfg['code']
    subtopics = dcfg['subtopics']
    
    for p_class in CLASSES:
        group_key = (domain, p_class)
        current = len(inventory[group_key])
        
        if current >= TARGET_PER_GROUP:
            print(f'[SKIP] {domain}/{p_class}: {current}/{TARGET_PER_GROUP}')
            continue
        
        needed = TARGET_PER_GROUP - current
        print(f'\n[GEN] {domain}/{p_class}: generating {needed} problems...')
        
        # Get reference exemplars for this group
        exemplars = ref_index.get(group_key, [])
        if not exemplars:
            # Fallback: use any items from same domain
            exemplars = [p for p in reference_data if p['domain'] == domain]
        
        # Plan difficulty-stratified batches
        subtopic_idx = current  # Continue rotation from where we left off
        
        while needed > 0:
            batch = min(BATCH_SIZE, needed)
            
            # Rotate difficulty tier
            tier_idx = (total_generated // 5) % len(DIFFICULTY_TIERS)
            difficulty = DIFFICULTY_TIERS[tier_idx]
            
            # Rotate sub-topic
            subtopic = subtopics[subtopic_idx % len(subtopics)]
            subtopic_idx += 1
            
            retries = 0
            while retries < RETRY_LIMIT:
                try:
                    prompt = build_prompt(domain, p_class, code, exemplars,
                                         batch, difficulty, subtopic)
                    raw = str(gen_llm.prompt(prompt))
                    items = extract_json_array(raw)
                    
                    accepted = []
                    for item in items:
                        if not validate_item(item):
                            continue
                        h = stable_hash(item['user'])
                        if h in existing_hashes:
                            continue
                        
                        # Normalize and stamp
                        ts = int(time.time() * 1000)
                        item['problem_id'] = f'GEN-{code}-{ts + len(accepted):013d}'
                        item['stable_hash'] = h
                        item['domain'] = domain
                        item['problem_class'] = p_class
                        solv_map = {'DETERMINATE':'Determinate', 'UNDERDETERMINED':'Underdetermined',
                                    'INSUFFICIENT':'Insufficient', 'CONTRADICTORY':'Contradictory'}
                        item['correct_solvability_class'] = solv_map[p_class]
                        item['system'] = SYSTEM_TEMPLATE
                        item['rigor_mode'] = 'base'
                        item['branching_factor'] = int(item.get('branching_factor', 2))
                        item['difficulty'] = difficulty[0]
                        item['subtopic'] = subtopic
                        item['generator'] = gen_model_name
                        
                        accepted.append(item)
                        existing_hashes.add(h)
                    
                    if accepted:
                        all_problems.extend(accepted)
                        inventory[group_key].extend(accepted)
                        needed -= len(accepted)
                        total_generated += len(accepted)
                        
                        # Checkpoint
                        with open(CHECKPOINT_PATH, 'w', encoding='utf-8') as f:
                            json.dump(all_problems, f)
                        
                        elapsed = time.time() - t_start
                        rate = total_generated / (elapsed / 60) if elapsed > 0 else 0
                        pct = len(all_problems) / total_target * 100
                        print(f'  +{len(accepted)} | Total: {len(all_problems)}/{total_target} ({pct:.0f}%) | '
                              f'{difficulty[0]:7s} | {subtopic} | {rate:.1f}/min')
                        break
                    else:
                        retries += 1
                        print(f'  [RETRY {retries}] No valid items in batch')
                
                except Exception as e:
                    retries += 1
                    total_errors += 1
                    print(f'  [ERROR {retries}] {type(e).__name__}: {str(e)[:120]}')
                    time.sleep(5)
            
            if retries >= RETRY_LIMIT:
                print(f'  [SKIP BATCH] Max retries, advancing...')
                needed -= batch
            
            time.sleep(COOLDOWN_SECS)

elapsed_total = time.time() - t_start
print(f'\n{"═"*70}')
print(f'  GENERATION COMPLETE')
print(f'  Total problems:    {len(all_problems)}')
print(f'  Generated this run:{total_generated}')
print(f'  Errors:            {total_errors}')
print(f'  Time:              {elapsed_total/60:.1f} minutes')
print(f'{"═"*70}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [C07] Scientific Validation Audit
# ══════════════════════════════════════════════════════════════════════════════

print('╔══════════════════════════════════════════════════════════════╗')
print('║         PROMETHEUS-EBM Dataset Validation Report           ║')
print('╚══════════════════════════════════════════════════════════════╝')
print(f'\nTotal items: {len(all_problems)}\n')

# 1. Matrix balance
print('─── Matrix Balance ───')
dist = Counter((p['domain'], p['problem_class']) for p in all_problems)
print(f'{"Domain":15s} │ {"Class":20s} │ Count │ Status')
print('─' * 65)
all_balanced = True
for domain in DOMAIN_CONFIG:
    for p_class in CLASSES:
        count = dist.get((domain, p_class), 0)
        status = '✓' if count >= TARGET_PER_GROUP else f'⚠️ NEED {TARGET_PER_GROUP - count} MORE'
        if count < TARGET_PER_GROUP:
            all_balanced = False
        print(f'{domain:15s} │ {p_class:20s} │ {count:5d} │ {status}')
print(f'\nMatrix balance: {"PASS ✓" if all_balanced else "INCOMPLETE"}')

# 2. Difficulty distribution
print('\n─── Difficulty Distribution ───')
diff_dist = Counter(p.get('difficulty', 'unknown') for p in all_problems)
for tier, count in sorted(diff_dist.items()):
    pct = count / len(all_problems) * 100
    print(f'  {tier:10s}: {count:4d} ({pct:.1f}%)')

# 3. Sub-topic coverage
print('\n─── Sub-topic Coverage ───')
for domain in DOMAIN_CONFIG:
    subtopics_seen = set(p.get('subtopic', '?') for p in all_problems if p['domain'] == domain)
    total_subs = len(DOMAIN_CONFIG[domain]['subtopics'])
    print(f'  {domain:15s}: {len(subtopics_seen)}/{total_subs} sub-topics covered')

# 4. Schema integrity
print('\n─── Schema Integrity ───')
required_keys = ['problem_id','stable_hash','domain','problem_class',
                 'correct_solvability_class','system','user','ground_truth_answer',
                 'branching_factor','rigor_mode']
schema_errors = 0
for i, p in enumerate(all_problems):
    for k in required_keys:
        if k not in p:
            schema_errors += 1
            if schema_errors <= 5:
                print(f'  ERROR: Item {i} ({p.get("problem_id","?")}) missing "{k}"')
print(f'Schema check: {"PASS ✓" if schema_errors == 0 else f"FAIL — {schema_errors} errors"}')

# 5. Deduplication
print('\n─── Deduplication ───')
user_texts = [p['user'] for p in all_problems]
unique = len(set(user_texts))
dupes = len(user_texts) - unique
# Also check overlap with original 200
ref_texts = {p['user'] for p in reference_data}
overlap = sum(1 for t in user_texts if t in ref_texts)
print(f'  Internal duplicates: {dupes}')
print(f'  Overlap with original 200: {overlap}')
print(f'  Dedup check: {"PASS ✓" if dupes == 0 and overlap == 0 else "WARN"}')

# 6. Length statistics
print('\n─── Length Statistics ───')
q_lens = [len(p['user']) for p in all_problems]
a_lens = [len(p['ground_truth_answer']) for p in all_problems]
print(f'  Questions: min={min(q_lens)}, max={max(q_lens)}, avg={sum(q_lens)/len(q_lens):.0f} chars')
print(f'  Answers:   min={min(a_lens)}, max={max(a_lens)}, avg={sum(a_lens)/len(a_lens):.0f} chars')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [C08] Export Final Lab-Grade Dataset
# ══════════════════════════════════════════════════════════════════════════════

# Sort: domain → class → difficulty → problem_id
domain_order = {d: i for i, d in enumerate(DOMAIN_CONFIG)}
class_order = {c: i for i, c in enumerate(CLASSES)}
diff_order = {'routine': 0, 'advanced': 1, 'expert': 2}

all_problems.sort(key=lambda p: (
    domain_order.get(p['domain'], 99),
    class_order.get(p['problem_class'], 99),
    diff_order.get(p.get('difficulty',''), 99),
    p['problem_id']
))

# Write final output
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_problems, f, indent=2, ensure_ascii=False)

file_size = os.path.getsize(OUTPUT_PATH)

print(f'\n{"═"*70}')
print(f'  ✓ FINAL DATASET EXPORTED')
print(f'  File:  {OUTPUT_PATH}')
print(f'  Size:  {file_size / 1024:.1f} KB ({file_size / 1024 / 1024:.2f} MB)')
print(f'  Items: {len(all_problems)}')
print(f'  Model: {gen_model_name}')
print(f'{"═"*70}')
print(f'\n→ Download from the notebook OUTPUT tab.')
print(f'→ Upload to your Kaggle dataset as prometheus_1000_dataset.json')
print(f'→ Use with Final_V5.ipynb in DEEP_PROBE mode for single-model diagnostics.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [C09] Quality Preview — Sample Problems by Class
# ══════════════════════════════════════════════════════════════════════════════

print('╔══════════════════════════════════════════════════════════════╗')
print('║            Sample Problems from Each Class                 ║')
print('╚══════════════════════════════════════════════════════════════╝\n')

for p_class in CLASSES:
    class_items = [p for p in all_problems if p['problem_class'] == p_class]
    if not class_items:
        continue
    # Pick one from each difficulty tier
    for tier_name, _, _ in DIFFICULTY_TIERS:
        tier_items = [p for p in class_items if p.get('difficulty') == tier_name]
        if tier_items:
            sample = random.choice(tier_items)
            print(f'── {p_class} / {tier_name.upper()} / {sample["domain"]} ({sample.get("subtopic","?")}) ──')
            print(f'Q: {sample["user"][:250]}')
            print(f'A: {sample["ground_truth_answer"][:200]}')
            print()
            break  # One per class for brevity

print('\n✓ Dataset generation complete. Ready for DEEP_PROBE evaluation.')